In [4]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

In [7]:
@dataclass
class GPTConfig:
    block_size: int = 256 #T
    vocab_size: int = 65
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384 #C
    dropout: float = 0.1

In [ ]:
class Head(nn.Module):
    """One head of causal self-attention.

    Maps (B, T, C = n_embd) -> (B, T, head_size). Each token forms a query, scores it
    against every past token's key, softmaxes into weights, and takes the
    weighted sum of their values.
    """
    def __init__(self, config: GPTConfig, head_size: int):
        super().__init__()
        self.head_size = head_size
        # key / query / value linear projections (n_embd -> head_size, no bias)
        self.query = nn.Linear(config.n_embd, head_size, bias=False)
        self.key = nn.Linear(config.n_embd, head_size, bias=False)
        self.value = nn.Linear(config.n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(config.block_size, config.block_size))) # T,T
        self.dropout = nn.Dropout(config.dropout)
        
    def forward(self, x):
        B, T, C = x.shape
        q = self.query(x) #(B, T, n_embd) -> (B, T, head_size)
        k = self.key(x)   #(B, T, n_embd) -> (B, T, head_size)
        v = self.value(x) #(B, T, n_embd) -> (B, T, head_size)
        
        # attention scores = q @ k^T scaled by 1/sqrt(head_size) -> BTT
        wei = q @ k.transpose(-2, -1) * self.head_size**-0.5 
        # mask future positions with tril (set to -inf), softmax, dropout
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) 
        wei = F.softmax(wei, dim=-1) # (B T T)
        wei = self.dropout(wei)

        out = wei @ v # (B T T) @ (B, T, head_size) ->   (B, T, head_size) 
        return out 
    
class MultiHeadAttention(nn.Module):
    """Several attention heads in parallel, concatenated and projected back."""    
    def __init__(self, config: GPTConfig):
        super().__init__()
        head_size = config.n_embd // config.n_head
        self.n_head = config.n_head        
        self.head_size = head_size
    
        self.heads = nn.ModuleList([Head(config, self.head_size) for _ in range(self.n_head)])
        self.proj = nn.Linear(self.n_head * self.head_size, config.n_embd) # output projection (n_embd -> n_embd) and dropout
        self.dropout = nn.Dropout(config.dropout)
    
    def forward(self, x):
        # concat all heads along the channel dim, then project + dropout
        # (B, T, head_size) -> (B, T, head_size * n_head) == (B, T, n_embd)
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.dropout(self.proj(out))

In [ ]:
class CausalSelfAttention(nn.Module):
    
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer('bias', torch.tril(torch.ones(config.block_size, config.block_size))
                             .view(1, 1, config.block_size, config.block_size))
        
    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) #(B, nh, T, hs)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) #(B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) #(B, nh, T, hs)
        
        att = q @ k.transpose(-2, -1) * (1.0 / k.size(-1)**-0.5)
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v 
        y = y.transpose(1,2).contiguous().view(B,T,C)
        y = self.c_proj(y) # output projection
        return y 
        
        
        
        
        
          

In [ ]:

class CausalSelfAttention(nn.Module):
    
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0        
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd) # key, query, Value projections for all heads        
        self.c_proj = nn.Linear(config.n_embd, config.n_embd) # projection
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer('bias', torch.tril(torch.ones(config.block_size, config.block_size))
                             .view(1, 1, config.block_size, config.block_size)) # (T, T) ->  (1,1,T,T)
        
    def forward(self, x):
        B, T, C = x.size() # (B, T, n_embd)
        
        qkv = self.c_attn(x) # (B, T, 2304(3 * n_embd))
        q, k, v = qkv.split(self.n_embd, dim=2) #3 * (B, T, 768(n_embd))) split in dim2, which is C dim         
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) #(B, nh, T, hs) (B, 12, T, 64)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) #(B, nh, T, hs) (B, 12, T, 64)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) #(B, nh, T, hs) (B, 12, T, 64)

        # (B,nh,T,hs) @ (B,nh,hs,T) -> (B,nh,T,T), then sqrt(hs)
        att = q @ k.transpose(-2, -1) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1) # (B,nh,T,T)
        y = att @ v # (B,nh,T,T) @ (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y) # output projection
        return y
